In [13]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/A.zip" /content/
!unzip -q /content/A.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/A/Development/en/corpus/10004? [y]es, [n]o, [A]ll, [N]one, [r]ename: None


In [14]:
ORIGINAL_TRAIN_CSV = "/content/A/Training/en/job_applicant_dataset_initial.csv"

FORMATTED_TRAIN_CSV = "/content/A/Training/en/job_applicant_dataset.csv"

DEV_CORPUS_DIR = "/content/A/Development/en/corpus"
DEV_QUERIES_DIR = "/content/A/Development/en/queries"

OUTPUT_DIR = "/content/dataset_prompt_validation_outputs"

MANUAL_SAMPLE_SIZE = 30


In [15]:
!pip install -q pandas numpy scikit-learn tqdm

In [16]:
import os
import re
import random
import shutil
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [17]:
EXPECTED_COLUMNS = [
    "Job Applicant Name",
    "Age",
    "Gender",
    "Race",
    "Ethnicity",
    "Resume",
    "Job Roles",
    "Job Description",
    "Best Match",
]

RESUME_COLUMN = "Resume"
JOB_ROLE_COLUMN = "Job Roles"
JOB_DESCRIPTION_COLUMN = "Job Description"
LABEL_COLUMN = "Best Match"


def check_schema(df, dataset_name):
    missing = [
        column for column in EXPECTED_COLUMNS
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{dataset_name} is missing these columns: {missing}"
        )

    return True


In [18]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "with", "on",
    "as", "by", "at", "from", "is", "are", "was", "were", "be", "this",
    "that", "it", "its", "their", "his", "her", "they", "we", "you",
    "candidate", "resume", "job", "description", "role", "position",
    "skills", "experience", "work", "working", "team", "teams",
}


def basic_tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-zA-Zăâîșşțţ0-9]+", text)


def content_terms(text):
    return {
        token
        for token in basic_tokenize(text)
        if len(token) >= 3 and token not in STOPWORDS
    }


def jaccard_text(a, b):
    set_a = content_terms(a)
    set_b = content_terms(b)

    if not set_a or not set_b:
        return 0.0

    return len(set_a & set_b) / len(set_a | set_b)


def normalize_label(value):
    text = str(value).strip().lower()

    if text in {"1", "true", "yes", "y", "match", "best", "best match"}:
        return "positive"
    if text in {"0", "false", "no", "n", "non-match", "not match", "not best match"}:
        return "negative"

    return text


def snippet(text, max_chars=300):
    value = str(text).replace("\n", " ").strip()
    value = re.sub(r"\s+", " ", value)

    if len(value) <= max_chars:
        return value

    return value[:max_chars] + "..."


def read_text_files(folder):
    if folder is None or not os.path.isdir(folder):
        return {}

    documents = {}

    for root, _, files in os.walk(folder):
        for filename in files:
            path = os.path.join(root, filename)

            try:
                with open(path, "r", encoding="utf-8-sig", errors="ignore") as file:
                    documents[os.path.relpath(path, folder)] = file.read()
            except Exception:
                continue

    return documents


def paired_tfidf_similarity(left_texts, right_texts):
    all_texts = list(left_texts) + list(right_texts)

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1
    )

    matrix = vectorizer.fit_transform(all_texts)
    left_matrix = matrix[:len(left_texts)]
    right_matrix = matrix[len(left_texts):]

    pair_scores = []

    for index in range(len(left_texts)):
        pair_scores.append(
            cosine_similarity(
                left_matrix[index],
                right_matrix[index]
            )[0, 0]
        )

    return pair_scores


def max_tfidf_similarity(source_texts, target_dict, source_label, target_label):
    if not target_dict:
        return pd.DataFrame()

    target_names = list(target_dict.keys())
    target_texts = [target_dict[name] for name in target_names]
    all_texts = list(source_texts) + target_texts

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1
    )

    matrix = vectorizer.fit_transform(all_texts)
    source_matrix = matrix[:len(source_texts)]
    target_matrix = matrix[len(source_texts):]

    similarities = cosine_similarity(source_matrix, target_matrix)
    best_indices = similarities.argmax(axis=1)
    best_scores = similarities.max(axis=1)

    return pd.DataFrame({
        source_label: np.arange(len(source_texts)),
        f"most_similar_{target_label}": [
            target_names[index] for index in best_indices
        ],
        "max_similarity": best_scores,
    })


In [19]:
if ORIGINAL_TRAIN_CSV is None or not os.path.exists(ORIGINAL_TRAIN_CSV):
    raise FileNotFoundError(
        "ORIGINAL_TRAIN_CSV must point to the original training file. "
        "This focused notebook needs both the original and formatted versions."
    )

if not os.path.exists(FORMATTED_TRAIN_CSV):
    raise FileNotFoundError(
        "FORMATTED_TRAIN_CSV does not exist. Check the configured path."
    )

original_df = pd.read_csv(ORIGINAL_TRAIN_CSV)
formatted_df = pd.read_csv(FORMATTED_TRAIN_CSV)

check_schema(original_df, "original dataset")
check_schema(formatted_df, "formatted dataset")

if len(original_df) != len(formatted_df):
    raise ValueError(
        "The original and formatted datasets do not have the same number of rows."
    )


In [20]:
comparison = pd.DataFrame({
    "row_id": np.arange(len(original_df)),
    "label_before": original_df[LABEL_COLUMN],
    "label_after": formatted_df[LABEL_COLUMN],
    "job_role_before": original_df[JOB_ROLE_COLUMN].astype(str),
    "job_role_after": formatted_df[JOB_ROLE_COLUMN].astype(str),
    "resume_before": original_df[RESUME_COLUMN].astype(str),
    "resume_after": formatted_df[RESUME_COLUMN].astype(str),
    "job_description_before": original_df[JOB_DESCRIPTION_COLUMN].astype(str),
    "job_description_after": formatted_df[JOB_DESCRIPTION_COLUMN].astype(str),
})

comparison["resume_before_after_jaccard"] = [
    jaccard_text(before, after)
    for before, after in zip(
        comparison["resume_before"],
        comparison["resume_after"]
    )
]

comparison["job_before_after_jaccard"] = [
    jaccard_text(before, after)
    for before, after in zip(
        comparison["job_description_before"],
        comparison["job_description_after"]
    )
]

comparison["resume_job_overlap_before"] = [
    jaccard_text(resume, job_description)
    for resume, job_description in zip(
        comparison["resume_before"],
        comparison["job_description_before"]
    )
]

comparison["resume_job_overlap_after"] = [
    jaccard_text(resume, job_description)
    for resume, job_description in zip(
        comparison["resume_after"],
        comparison["job_description_after"]
    )
]

comparison["resume_job_overlap_change"] = (
    comparison["resume_job_overlap_after"]
    - comparison["resume_job_overlap_before"]
)

comparison["resume_before_after_tfidf"] = paired_tfidf_similarity(
    comparison["resume_before"].tolist(),
    comparison["resume_after"].tolist()
)

comparison["job_before_after_tfidf"] = paired_tfidf_similarity(
    comparison["job_description_before"].tolist(),
    comparison["job_description_after"].tolist()
)

overlap_metrics = [
    "resume_before_after_jaccard",
    "job_before_after_jaccard",
    "resume_job_overlap_before",
    "resume_job_overlap_after",
    "resume_job_overlap_change",
    "resume_before_after_tfidf",
    "job_before_after_tfidf",
]

overlap_summary = []

for metric in overlap_metrics:
    values = comparison[metric].replace([np.inf, -np.inf], np.nan).dropna()

    overlap_summary.append({
        "metric": metric,
        "mean": values.mean(),
        "median": values.median(),
        "min": values.min(),
        "max": values.max(),
    })

overlap_summary = pd.DataFrame(overlap_summary)

display(overlap_summary)

,metric,mean,median,min,max
0,resume_before_after_jaccard,0.095211,0.097701,0.045977,0.138365
1,job_before_after_jaccard,0.562069,0.564706,0.482759,0.638298
2,resume_job_overlap_before,0.056948,0.055556,0.000000,0.142857
3,resume_job_overlap_after,0.089405,0.089362,0.047619,0.139241
4,resume_job_overlap_change,0.032457,0.033924,-0.070085,0.126016
5,resume_before_after_tfidf,0.492283,0.487392,0.283019,0.764237
6,job_before_after_tfidf,0.949716,0.954793,0.815604,0.984098


In [21]:
dev_corpus_docs = read_text_files(DEV_CORPUS_DIR)
dev_query_docs = read_text_files(DEV_QUERIES_DIR)

resume_dev_similarity = max_tfidf_similarity(
    comparison["resume_after"].tolist(),
    dev_corpus_docs,
    "row_id",
    "dev_resume"
)

job_dev_similarity = max_tfidf_similarity(
    comparison["job_description_after"].tolist(),
    dev_query_docs,
    "row_id",
    "dev_query"
)

high_resume_similarity = (
    resume_dev_similarity[
        resume_dev_similarity["max_similarity"] >= 0.85
    ]
    if not resume_dev_similarity.empty
    else pd.DataFrame()
)

high_job_similarity = (
    job_dev_similarity[
        job_dev_similarity["max_similarity"] >= 0.85
    ]
    if not job_dev_similarity.empty
    else pd.DataFrame()
)

development_similarity_summary = pd.DataFrame({
    "check": [
        "formatted_resumes_above_0_85",
        "formatted_job_descriptions_above_0_85",
        "max_resume_development_similarity",
        "max_job_development_similarity",
    ],
    "value": [
        len(high_resume_similarity),
        len(high_job_similarity),
        (
            resume_dev_similarity["max_similarity"].max()
            if not resume_dev_similarity.empty else np.nan
        ),
        (
            job_dev_similarity["max_similarity"].max()
            if not job_dev_similarity.empty else np.nan
        ),
    ],
})

display(development_similarity_summary)

,check,value
0,formatted_resumes_above_0_85,0.000000
1,formatted_job_descriptions_above_0_85,0.000000
2,max_resume_development_similarity,0.715671
3,max_job_development_similarity,0.529491
